# Typst PDF generation

Fecha: 22-mayo-205

Objetivo: Generar archivos de PDF utilizando la libreria Typst

In [6]:
from importlib import reload

from eerssa import gestionOT
from eerssa import matrizActividades
from eerssa import reporte_ot as reporte_pdf

import typst
import pypst
import pickle
from pprint import pprint

reload(reporte_pdf)


<module 'eerssa.reporte_ot' from '/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/eerssa/reporte_ot.py'>

In [2]:
ot = "/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/tests/db_test/2025-04-ABRIL_1_object.pkl"

with open(ot, 'rb') as f:
    obj_lists = pickle.load(f)

type(obj_lists)


list

In [3]:
logging_info = obj_lists[50].log
logging_info[0]

{'t': '2025-05-04T12:51:54.298773',
 'level': 'INFO',
 'message': 'CREACIÓN DE LA OT, se encuentra un archivo PDF de al menos tres hojas ',
 'detail': 'Ninguno'}

In [7]:
test = reporte_pdf.create_typst_doc( obj_lists[50])
pprint(test.render())

('#import "eerssa/templateReporte.typ": *\n'
 '\n'
 '#show: dvdtyp.with(\n'
 '    title: "Reporte de Orden de Trabajo",\n'
 '    subtitle: [ ],\n'
 '    author: "TERMINADO",\n'
 '    abstract: "Zamora Z1 (Cuadrilla. Nro. 6) \n'
 ' sábado, 26 de abril del 2025 \n'
 ' SILVA ARMIJOS ROMEL EDUARDO",\n'
 '  )\n'
 '\n'
 '= Novedades encontradas\n'
 '\n'
 '#informativo("Informativo - 2025-05-04 12:51:54")[\n'
 '  CREACIÓN DE LA OT, se encuentra un archivo PDF de al menos tres hojas \n'
 '  $\n'
 '    "Ninguno"\n'
 '  $\n'
 ']\n'
 '\n'
 '#error("ERROR - 2025-05-04 12:53:06")[\n'
 '  No coinciden las fechas\n'
 '  $\n'
 '    "La fecha en Hoja 1 no es la misma que en Actividades"\n'
 '  $\n'
 ']')


In [8]:
with open("reporte.typ", mode="wt") as f:
    f.write(test.render())

typst.compile("reporte.typ",  output="Reporte_OT.pdf")

## DEBUG

In [26]:
# Si suponemos ejecuación de OTs validas, deberia haber esta información como mínimo
# para cualquier otro caso debe funcionar el EXCEPT

ot_nro = 50

try:
  estado = obj_lists[ot_nro].data["estado"]
except:
  estado = "SIN ESTADO"
try:
  cuadrilla = obj_lists[ot_nro].data["cuadrilla"]
except:
  cuadrilla = "SIN CUADRILLA"
try:
  fecha = obj_lists[ot_nro].data["fechaInicio"]
except:
  fecha = "Sin Fecha"
try:
  responsable = obj_lists[ot_nro].data["responsable"][0]
except:
  responsable = "Sin Responsable"

In [ ]:
doc = pypst.Document()
doc.add_import("notebooks/templateReporte.typ", ['*']) # Import all Modules


doc.add(f"""#show: dvdtyp.with(
  title: "Reporte de Orden de Trabajo",
  subtitle: [ ],
  author: "{estado}",
  abstract: "{cuadrilla} \n {fecha} \n {responsable}",
)""")

doc.add("= Novedades encontradas")
pprint(doc.render())

('#import "notebooks/templateReporte.typ": *\n'
 '\n'
 '#show: dvdtyp.with(\n'
 '  title: "Reporte de Orden de Trabajo",\n'
 '  subtitle: [ ],\n'
 '  author: "TERMINADO",\n'
 '  abstract: "Zamora Z1 (Cuadrilla. Nro. 6) \n'
 ' sábado, 26 de abril del 2025 \n'
 ' SILVA ARMIJOS ROMEL EDUARDO",\n'
 ')\n'
 '\n'
 '== Novedades encontradas')


In [75]:
def generate_comment( doc, log_item ):
  level_msg = log_item["level"]
  time_msg = log_item["t"].replace('T',' ').split('.')[0]
  detail = log_item["detail"]
  message = log_item["message"]
  
  if level_msg == 'FATAL':
    color = f"#problem(\"FATAL - {time_msg}\")"
  elif level_msg == 'ERROR':
    color = f"#error(\"ERROR - {time_msg}\")"
  elif level_msg == 'REVISAR':
    color = f"#revisar(\"Revisar - {time_msg}\")"
  else:
    color = f"#informativo(\"Informativo - {time_msg}\")"

  doc.add(
    f"""{color}[
  {message}
  $
    \"{detail}"
  $
]""")

  return doc

In [76]:
generate_comment(doc, logging_info[1])
pprint(doc.render())

('#import "notebooks/templateReporte.typ": *\n'
 '\n'
 '#show: dvdtyp.with(\n'
 '  title: "Reporte de Orden de Trabajo",\n'
 '  subtitle: [ ],\n'
 '  author: "TERMINADO",\n'
 '  abstract: "Zamora Z1 (Cuadrilla. Nro. 6) \n'
 ' sábado, 26 de abril del 2025 \n'
 ' SILVA ARMIJOS ROMEL EDUARDO",\n'
 ')\n'
 '\n'
 '== Novedades encontradas\n'
 '\n'
 '#error("ERROR - 2025-05-04 12:53:06")[\n'
 '  No coinciden las fechas\n'
 '  $\n'
 '    "La fecha en Hoja 1 no es la misma que en Actividades"\n'
 '  $\n'
 ']')


In [77]:
with open("Typst_example.typ", mode="wt") as f:
    f.write(doc.render())

typst.compile("Typst_example.typ",  output="Reporte_OT.pdf")